# Evaluation

For my evaluation, I will compare base Phi4 performance (no fine-tuning) with the best checkpoint fine-tuned on 3k examples and 20k examples. For each, I calculate the following performance metrics:
- Accuracy: the proportion of correct predictions (both positive and negative) out of the total number of cases examined
- Recall: the identification all relevant instances (positive cases) within a dataset
- F1: the harmonic mean of precision and recall

For my project, **recall** is the most important metric, as it is imperative that PII does not get leaked through data sharing. While over-redacting may impair the usefulness of the data, under-redacting has significant consequences for the research lab and fails to protect individuals' privacy.

## Setup

First, I am importing all of the necessary libraries for the evalutation. Since the evaluation does use `FastLanguageModel` from `unsloth` to speed up model loading, it does require GPU compute. However, if you would like to run this on CPU, you can modify the way the model is loaded. 

In [1]:
import re
import json
import glob
import json
import os
import torch
import warnings
from unsloth import FastLanguageModel
from pathlib import Path
from sklearn.metrics import classification_report, f1_score
from collections import defaultdict
import numpy as np
import pandas as pd
from datasets import load_dataset, Dataset, concatenate_datasets, load_from_disk

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


For visibility, I am defining my system prompt and the relevant categories of PII below. 

In [2]:
SYSTEM_PROMPT = """
You are a personally identifiable information (PII) redaction assistant. Wrap all personally identifiable
information in XML-style tags sych as <NAME>, <DOB>, <PHONE, <ADDRESS>, <LOCATION> etc. 
Rules:
- Return the FULL transcript with PII tagged.
- Do NOT change any non-PII text.
- If no PII is present, return the transcript unchanged.
"""

TAG_COLS = [
    'NAME', 'TITLE', 'DOB', 'DATE', 'TIME', 'EMAIL', 'PHONE',
    'ADDRESS', 'LOCATION', 'SSN', 'UNIQUE_ID', 'LICENSE_NUMBER',
    'IP_ADDRESS', 'USERNAME', 'PASS', 'SEX', 'CARDISSUER'
]

Next, I am loading the dataset. You made load the models that I pretrained or you may perform the training yourself prior to running this notebook.

In [53]:
## Load datasets
val_300k_dataset = load_from_disk("../data/ai4privacy/pii-masking-300k_val_1500")
np_silver_dataset = load_from_disk("../data/np_silver/np_silver_100_processed")

In [5]:
## print the datasets to verify they loaded correctly
print(val_300k_dataset)
print(np_silver_dataset)

Dataset({
    features: ['source_text', 'contains_pii', 'pii_tags', 'ai4privacy_text', 'tagged_text', 'NAME', 'TITLE', 'DOB', 'DATE', 'TIME', 'EMAIL', 'PHONE', 'ADDRESS', 'LOCATION', 'SSN', 'UNIQUE_ID', 'LICENSE_NUMBER', 'IP_ADDRESS', 'USERNAME', 'PASS', 'SEX', 'CARDISSUER'],
    num_rows: 1500
})
Dataset({
    features: ['source_text', 'contains_pii', 'pii_tags', 'ai4privacy_text', 'tagged_text', 'NAME', 'TITLE', 'DOB', 'DATE', 'TIME', 'EMAIL', 'PHONE', 'ADDRESS', 'LOCATION', 'SSN', 'UNIQUE_ID', 'LICENSE_NUMBER', 'IP_ADDRESS', 'USERNAME', 'PASS', 'SEX', 'CARDISSUER'],
    num_rows: 1500
})


The datasets must contain the columns `source_text`, `contains_pii`, `tagged_text`, and each of the types of PII tags for the code below. 

## Defining Functions for Inference and Evaluation
For visibility, I have left each of these functions defined in this notebook, but they may also be found in the scripts folder of this repo. I recommend importing them from scripts if you choose to use them in your own work.

The scripts here include:
- `llm_inference()`: This passes each text in the correct format to the LLM and asks the model to generate. It then decodes the model's response and adds its response to a list of texts, which is returned.
- `run_inference()`: This wraps llm_inference to handle logging, resuming from previous runs, and aligning generated text with input data. 

In [7]:
def llm_inference(model, tokenizer, texts, sys_prompt=SYSTEM_PROMPT, max_new_tokens=512, debug=False):
    """
    Generate redacted/tagged output from a list of texts.
    Returns list of generated text
    """
    results = []

    tokenizer.padding_side = "left"
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    if debug:
        print('texts recieved')
        print(texts)
    for text in texts:
        # Matches prompt format used during fine-tuning
        messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user",   "content": text.strip()},
        ]
        prompt = tokenizer.apply_chat_template(
            messages,
            tokenize              = False,
            add_generation_prompt = True, # ensures the answer is not included
        )

        inputs = tokenizer(prompt, 
                           return_tensors="pt",
                           padding=True, # pad shorter sequences to match longest in batch
                           truncation=True
                          ).to(model.device)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                pad_token_id=tokenizer.eos_token_id,
                do_sample=False,  # greedy so that it is deterministic
            )
        
        # Decode only the newly generated tokens and ignore the 
        generated = tokenizer.decode(
            outputs[0][inputs["input_ids"].shape[1]:],
            skip_special_tokens=True
        ).strip()
        
        results.append(generated)
        if debug:
            print('generated:')
            print(generated)
        
    return results

In [10]:
def run_inference(val_dataset, model, tokenizer, 
                       batch_size=4, max_new_tokens=512, tagged_key="tagged_text", save_path="../data/ft_predictions.jsonl", debug=False):
    """
    Keeps track of the inferences run for a model and logs progress. It calls llm_inference()
    Use debug mode if you are running into issues (set debug=True)
    Returns the predictions associated with their tags, source text, and ground truth text
    """
    already_done = 0
    all_predictions = []
    # Checking to see if we should resume from a previous run 
    if save_path and os.path.exists(save_path):
        with open(save_path, "r") as f:
            lines = [line.strip() for line in f if line.strip()]
        already_done = len(lines)
        all_predictions = [json.loads(line) for line in lines]
        print(f"Resuming from record {already_done}/{len(val_dataset)}")
    else:
        print(f"Starting fresh: {len(val_dataset)} samples to process")

    for i in range(already_done, len(val_dataset), batch_size):
        batch = val_dataset.select(range(i, min(i + batch_size, len(val_dataset))))
        texts = batch["source_text"]
        
        generated_outputs = llm_inference(model, tokenizer, texts, 
                                          max_new_tokens=max_new_tokens)
        if debug:
            print("GENERATED OUTPUTS")
            print(generated_outputs)
        # associate with data
        for generated, row in zip(generated_outputs, batch):
            record = {
                "source_text": row["source_text"],
                "tagged_text": row[tagged_key],
                "generated": generated,
                "contains_pii": row["contains_pii"],
                **{tag: row[tag] for tag in TAG_COLS}
            }
            all_predictions.append(record)

        # Save incrementally — safe against crashes
        if save_path:
            with open(save_path, "a") as f:
                for record in all_predictions[-batch_size:]:
                    f.write(json.dumps(record) + "\n")

        # print progress
        print(f"{min(i + batch_size, len(val_dataset))}/{len(val_dataset)} complete", end="\r")

    print(f"Inference complete. Saved to {save_path}")
    return all_predictions

## Loading the models

I chose to suppress the print statements for the model loading because they were quite long. Please unsuppress by removing `%%capture` if you run into issues.

In [11]:
%%capture
ft_3k_model, ft_3k_tokenizer = FastLanguageModel.from_pretrained(
        model_name     = "../checkpoints/checkpoint-750",
        max_seq_length = 2048,
        dtype          = None,
        load_in_4bit   = True,
        cache_dir      = "/scratch/mlong6/huggingface_cache",
)
FastLanguageModel.for_inference(ft_3k_model)

Unsloth 2026.4.8 patched 40 layers with 0 QKV layers, 0 O layers and 0 MLP layers.


In [12]:
%%capture
ft_20k_model, ft_20k_tokenizer = FastLanguageModel.from_pretrained(
        model_name     = "../checkpoints_20000/checkpoint-10000",
        max_seq_length = 2048,
        dtype          = None,
        load_in_4bit   = True,
        cache_dir      = "/scratch/mlong6/huggingface_cache",
)
FastLanguageModel.for_inference(ft_20k_model)

Unsloth 2026.4.8 patched 40 layers with 40 QKV layers, 40 O layers and 40 MLP layers.


In [13]:
%%capture
base_model, base_tokenizer = FastLanguageModel.from_pretrained(
        model_name     ="unsloth/phi-4",
        max_seq_length = 2048,
        dtype          = None,
        load_in_4bit   = True,
        cache_dir      = "/scratch/mlong6/huggingface_cache",
)
FastLanguageModel.for_inference(base_model)

### Check GPU Usage before evaluation
I was curious about GPU usage, so I ran this before and after the evaluation. Please use these calculations when deciding how much compute you need.

In [14]:
## Checking GPU Usage before
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA A40. Max memory = 44.422 GB.
29.596 GB of memory reserved.


## Run Evaluation

I previously ran the evaluation, so these cells will run quickly. However, the initial run on 1500 prompts will take between 3-4 hours, and running on 100 prompts will take 20-30 minutes. 

### Evaluation Scripts

I also included the evaluation scripts here for visibility, but please import them from the scripts folder for your own use. The evaluation scripts below include:

- `compute_tag_metrics()`: evaluate the script at the tag level. Correct if the tag appears in both the response and the ground truth
- `print_tag_report()`: prints the results from compute_tag_metrics

In [28]:
def print_tag_report(results):
    print("=" * 60)
    print("PER-TAG PERFORMANCE")
    print("=" * 60)
    print(f"{'Tag':<20} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print("-" * 60)

    all_f1s = []
    for tag in TAG_COLS:
        tp = results[tag]["tp"]
        fp = results[tag]["fp"]
        fn = results[tag]["fn"]
        support = tp + fn

        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        all_f1s.append(f1)

        print(f"  {tag:<18} {precision:>10.3f} {recall:>10.3f} {f1:>10.3f} {support:>10}")

    print("-" * 60)
    print(f"{'Macro F1':<20} {np.mean(all_f1s):>10.3f}")
    print("=" * 60)

In [29]:
def compute_tag_metrics(predictions, tag_syntax="<{tag}>"):
    """
    Evaluation at the tag level
    tag_syntax: adjust to match your training format e.g. '<{tag}>' or '[{tag}]'
    """
    results = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})

    for record in predictions:
        predicted_tags = set()
        for tag in TAG_COLS:
            if tag_syntax.format(tag=tag) in record["generated"]:
                predicted_tags.add(tag)

        for tag in TAG_COLS:
            label = record[tag]
            predicted = tag in predicted_tags
            if label and predicted:
                results[tag]["tp"] += 1
            elif not label and predicted:
                results[tag]["fp"] += 1
            elif label and not predicted:
                results[tag]["fn"] += 1

    print_tag_report(results)
    return results

### Evaluation Tier 1

First, I am running on a validation set from `ai4privacy/pii-masking-300k`. This is expected to perform better because is from the same original dataset as the training data.

In [18]:
warnings.filterwarnings('ignore')
base_val_results = run_inference(val_300k_dataset, base_model, base_tokenizer, save_path="../data/base_val_1500.jsonl")
ft_3k_val_results = run_inference(val_300k_dataset, ft_3k_model, ft_3k_tokenizer, save_path="../data/finetuned_3k_val_1500.jsonl")
ft_20k_val_results = run_inference(val_300k_dataset, ft_20k_model, ft_20k_tokenizer, save_path="../data/finetuned_20k_val_1500.jsonl")

Resuming from record 1500/1500
Inference complete. Saved to ../data/base_val_1500.jsonl
Resuming from record 1500/1500
Inference complete. Saved to ../data/finetuned_3k_val_1500.jsonl
Resuming from record 1500/1500
Inference complete. Saved to ../data/finetuned_20k_val_1500.jsonl


### Calculate tag-level results
I first caluculated tag-level results because it was the easiest way to evaluate the data for inital metrics. These are the metrics that I presented in class, however with a smaller evaluation dataset size (500 vs 1500 in this evaluation). I am including it here to show my progress, but I learned that this metric is not particularly meaningful, as it checks that a specific tag exists at all within a generated text rather than if it occurs for the correct entities.

#### Base Phi4 Results

In [32]:
compute_tag_metrics(base_val_results)

PER-TAG PERFORMANCE
Tag                   Precision     Recall         F1    Support
------------------------------------------------------------
  NAME                    0.497      0.703      0.582        360
  TITLE                   0.727      0.048      0.091        165
  DOB                     0.728      0.873      0.794        205
  DATE                    0.639      0.370      0.468        230
  TIME                    0.773      0.048      0.091        352
  EMAIL                   0.796      0.849      0.822        225
  PHONE                   0.555      0.856      0.673        194
  ADDRESS                 0.709      0.340      0.460        309
  LOCATION                0.504      0.566      0.534        316
  SSN                     0.917      0.446      0.600        222
  UNIQUE_ID               0.000      0.000      0.000        355
  LICENSE_NUMBER          1.000      0.014      0.028        208
  IP_ADDRESS              0.966      0.425      0.590        200
  USERNAM

defaultdict(<function __main__.compute_tag_metrics.<locals>.<lambda>()>,
            {'NAME': {'tp': 253, 'fp': 256, 'fn': 107},
             'DOB': {'tp': 179, 'fp': 67, 'fn': 26},
             'TIME': {'tp': 17, 'fp': 5, 'fn': 335},
             'DATE': {'tp': 85, 'fp': 48, 'fn': 145},
             'ADDRESS': {'tp': 105, 'fp': 43, 'fn': 204},
             'LOCATION': {'tp': 179, 'fp': 176, 'fn': 137},
             'SSN': {'tp': 99, 'fp': 9, 'fn': 123},
             'USERNAME': {'tp': 47, 'fp': 9, 'fn': 206},
             'TITLE': {'tp': 8, 'fp': 3, 'fn': 157},
             'UNIQUE_ID': {'tp': 0, 'fp': 0, 'fn': 355},
             'LICENSE_NUMBER': {'tp': 3, 'fp': 0, 'fn': 205},
             'PASS': {'tp': 3, 'fp': 0, 'fn': 130},
             'PHONE': {'tp': 166, 'fp': 133, 'fn': 28},
             'EMAIL': {'tp': 191, 'fp': 49, 'fn': 34},
             'IP_ADDRESS': {'tp': 85, 'fp': 3, 'fn': 115},
             'SEX': {'tp': 1, 'fp': 1, 'fn': 171},
             'CARDISSUER': {'tp': 0, 'f

#### Finetuned 3k Results

In [31]:
compute_tag_metrics(ft_3k_val_results)

PER-TAG PERFORMANCE
Tag                   Precision     Recall         F1    Support
------------------------------------------------------------
  NAME                    0.936      0.969      0.952        360
  TITLE                   0.893      0.958      0.924        165
  DOB                     0.957      0.966      0.961        205
  DATE                    0.918      0.930      0.924        230
  TIME                    0.927      0.977      0.952        352
  EMAIL                   0.987      0.991      0.989        225
  PHONE                   0.978      0.912      0.944        194
  ADDRESS                 0.977      0.974      0.976        309
  LOCATION                0.942      0.972      0.956        316
  SSN                     0.919      0.869      0.894        222
  UNIQUE_ID               0.898      0.997      0.945        355
  LICENSE_NUMBER          0.964      0.904      0.933        208
  IP_ADDRESS              1.000      0.990      0.995        200
  USERNAM

defaultdict(<function __main__.compute_tag_metrics.<locals>.<lambda>()>,
            {'NAME': {'tp': 349, 'fp': 24, 'fn': 11},
             'DOB': {'tp': 198, 'fp': 9, 'fn': 7},
             'TIME': {'tp': 344, 'fp': 27, 'fn': 8},
             'DATE': {'tp': 214, 'fp': 19, 'fn': 16},
             'ADDRESS': {'tp': 301, 'fp': 7, 'fn': 8},
             'LOCATION': {'tp': 307, 'fp': 19, 'fn': 9},
             'SSN': {'tp': 193, 'fp': 17, 'fn': 29},
             'USERNAME': {'tp': 236, 'fp': 16, 'fn': 17},
             'TITLE': {'tp': 158, 'fp': 19, 'fn': 7},
             'UNIQUE_ID': {'tp': 354, 'fp': 40, 'fn': 1},
             'LICENSE_NUMBER': {'tp': 188, 'fp': 7, 'fn': 20},
             'PHONE': {'tp': 177, 'fp': 4, 'fn': 17},
             'PASS': {'tp': 130, 'fp': 3, 'fn': 3},
             'EMAIL': {'tp': 223, 'fp': 3, 'fn': 2},
             'IP_ADDRESS': {'tp': 198, 'fp': 0, 'fn': 2},
             'SEX': {'tp': 171, 'fp': 11, 'fn': 1},
             'CARDISSUER': {'tp': 0, 'fp': 0, 'f

#### Finetuned 20k Results

In [34]:
compute_tag_metrics(ft_20k_val_results)

PER-TAG PERFORMANCE
Tag                   Precision     Recall         F1    Support
------------------------------------------------------------
  NAME                    0.981      0.981      0.981        360
  TITLE                   0.964      0.982      0.973        165
  DOB                     0.975      0.941      0.958        205
  DATE                    0.931      0.939      0.935        230
  TIME                    0.977      0.977      0.977        352
  EMAIL                   0.978      0.996      0.987        225
  PHONE                   0.990      0.974      0.982        194
  ADDRESS                 0.993      0.981      0.987        309
  LOCATION                0.960      0.984      0.972        316
  SSN                     0.977      0.955      0.966        222
  UNIQUE_ID               0.956      0.986      0.971        355
  LICENSE_NUMBER          0.980      0.952      0.966        208
  IP_ADDRESS              0.980      0.995      0.988        200
  USERNAM

defaultdict(<function __main__.compute_tag_metrics.<locals>.<lambda>()>,
            {'NAME': {'tp': 353, 'fp': 7, 'fn': 7},
             'DOB': {'tp': 193, 'fp': 5, 'fn': 12},
             'TIME': {'tp': 344, 'fp': 8, 'fn': 8},
             'DATE': {'tp': 216, 'fp': 16, 'fn': 14},
             'ADDRESS': {'tp': 303, 'fp': 2, 'fn': 6},
             'LOCATION': {'tp': 311, 'fp': 13, 'fn': 5},
             'SSN': {'tp': 212, 'fp': 5, 'fn': 10},
             'USERNAME': {'tp': 247, 'fp': 12, 'fn': 6},
             'TITLE': {'tp': 162, 'fp': 6, 'fn': 3},
             'UNIQUE_ID': {'tp': 350, 'fp': 16, 'fn': 5},
             'LICENSE_NUMBER': {'tp': 198, 'fp': 4, 'fn': 10},
             'PASS': {'tp': 131, 'fp': 3, 'fn': 2},
             'PHONE': {'tp': 189, 'fp': 2, 'fn': 5},
             'EMAIL': {'tp': 224, 'fp': 5, 'fn': 1},
             'IP_ADDRESS': {'tp': 199, 'fp': 4, 'fn': 1},
             'SEX': {'tp': 172, 'fp': 2, 'fn': 0},
             'CARDISSUER': {'tp': 0, 'fp': 0, 'fn': 0}}

Both finetuned models do noticeably better that the base model with the when evaluating based on tags. Noteably, the recall remains upward of 0.9, meaning that there is little PII going undetected versus lower recall across categories for Phi4 base model.

### Token-level results
More meaningfully, I decided to calculate token-level results to ensure the tokens being tagged were indeed PII when compared against the ground truth. This is a more accurate metric because it checks multiple occurrences, handles if tags are at the word or phrase level, and checks for hallucinations (tokens that were not in the original source text). Scripts include:

- `evaluate_token_level()`: evaluates the results by token-level matching to ensure that every token that is flagged as PII in the ground truth is also caught by the LLM
- `print_token_report()`:  prints out evaluate_token_level
- `print_failures()`: prints out the samples with the worst performance for manual review
- `tokenize_value()`: split a span value into individual tokens, lowercase
- `extract_tag_tokens()`: extract per-tag token sets from tagged text

In [41]:
def print_token_report(tag_results, overall, counts_dict):

    def prf(tp, fp, fn):
        precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        recall    = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0
        support   = tp + fn
        return precision, recall, f1, support

    print("=" * 65)
    print("Token-level (per-tag) Performance")
    print("(each PII token counted individually)")
    print("=" * 65)
    print(f"{'Tag':<20} {'Precision':>10} {'Recall':>10} {'F1':>10} {'Support':>10}")
    print("-" * 65)

    all_f1s = []
    for tag in TAG_COLS:
        r = tag_results[tag]
        if r["tp"] + r["fp"] + r["fn"] == 0:
            continue
        p, r_, f1, support = prf(r["tp"], r["fp"], r["fn"])
        all_f1s.append(f1)
        print(f"  {tag:<18} {p:>10.3f} {r_:>10.3f} {f1:>10.3f} {support:>10}")

    print("-" * 65)
    p, r_, f1, support = prf(overall["tp"], overall["fp"], overall["fn"])
    print(f"{'Overall':<20} {p:>10.3f} {r_:>10.3f} {f1:>10.3f} {support:>10}")
    print(f"{'Macro F1':<20} {np.mean(all_f1s):>10.3f}")
    print(f"\nHallucinated tokens (not in source): {counts_dict['total_non_matches']}")
    print("=" * 65)

In [42]:
def tokenize_value(value):
    """Split a span value into individual tokens, lowercase."""
    return set(w.lower() for w in re.split(r"[^a-zA-Z0-9]+", value) if w)


def extract_tag_tokens(text):
    """
    Extract per-tag token sets from tagged text.
    Returns dict of {tag: set(tokens)} so we can evaluate per-tag.
    Also returns a flat set of all PII tokens for overall metrics.
    """
    tag_tokens = defaultdict(set)
    
    for tag in TAG_COLS:
        pattern = rf'<{tag}>(.*?)<[/\\]{tag}>'
        matches = re.findall(pattern, text, re.IGNORECASE | re.DOTALL)
        for match in matches:
            tokens = tokenize_value(match)
            tag_tokens[tag].update(tokens)
    
    return tag_tokens

In [45]:
def evaluate_token_level(predictions, verbose=False):
    """
    Token-level evaluation
    Counts each individual token as TP/FP/FN 
    """
    tag_results  = defaultdict(lambda: {"tp": 0, "fp": 0, "fn": 0})
    overall      = {"tp": 0, "fp": 0, "fn": 0}
    counts_dict  = {"total_non_matches": 0}
    sample_details = []

    for i, record in enumerate(predictions):
        source_text_lower = record["source_text"].lower()

        gt_tag_tokens   = extract_tag_tokens(record["tagged_text"])
        pred_tag_tokens = extract_tag_tokens(record["generated"])

        # Validate predicted tokens exist in source text
        # (catches hallucinated PII — your non_match logic)
        cleaned_pred_tag_tokens = defaultdict(set)
        for tag, tokens in pred_tag_tokens.items():
            for token in tokens:
                if token not in source_text_lower:
                    counts_dict["total_non_matches"] += 1
                    if verbose:
                        print(f"Hallucinated token '{token}' not in source text")
                else:
                    cleaned_pred_tag_tokens[tag].add(token)

        sample_tp, sample_fp, sample_fn = 0, 0, 0
        missed_tokens = defaultdict(set)
        extra_tokens  = defaultdict(set)

        for tag in TAG_COLS:
            gt_tokens   = gt_tag_tokens.get(tag, set())
            pred_tokens = cleaned_pred_tag_tokens.get(tag, set())

            tp = len(gt_tokens & pred_tokens)   # correctly caught
            fp = len(pred_tokens - gt_tokens)   # predicted but not in GT
            fn = len(gt_tokens - pred_tokens)   # in GT but missed

            tag_results[tag]["tp"] += tp
            tag_results[tag]["fp"] += fp
            tag_results[tag]["fn"] += fn

            overall["tp"] += tp
            overall["fp"] += fp
            overall["fn"] += fn

            sample_tp += tp
            sample_fp += fp
            sample_fn += fn

            if fn > 0:
                missed_tokens[tag] = gt_tokens - pred_tokens
            if fp > 0:
                extra_tokens[tag]  = pred_tokens - gt_tokens

        sample_details.append({
            "index":         i,
            "source_text":   record.get("source_text", "")[:100],
            "tp":            sample_tp,
            "fp":            sample_fp,
            "fn":            sample_fn,
            "missed_tokens": dict(missed_tokens),
            "extra_tokens":  dict(extra_tokens),
        })

    print_token_report(tag_results, overall, counts_dict)
    return tag_results, overall, sample_details, counts_dict

In [43]:
def print_failures(sample_details, n=10):
    sorted_samples = sorted(sample_details, key=lambda x: x["fn"], reverse=True)
    print("\nWORST SAMPLES (most missed PII tokens)\n")
    for sample in sorted_samples[:n]:
        if not sample["missed_tokens"] and not sample["extra_tokens"]:
            continue
        print(f"Sample {sample['index']}: {sample['source_text']}...")
        print(f"Missed(FN): {dict(sample['missed_tokens'])}")
        print(f"Extra(FP): {dict(sample['extra_tokens'])}")
        print()

#### Base Phi4 Results

In [46]:
base_tag_results, base_overall, base_sample_details, base_counts_dict = evaluate_token_level(base_val_results)

Token-level (per-tag) Performance
(each PII token counted individually)
Tag                   Precision     Recall         F1    Support
-----------------------------------------------------------------
  NAME                    0.589      0.505      0.544       1118
  TITLE                   0.959      0.378      0.542        437
  DOB                     0.810      0.723      0.764        959
  DATE                    0.481      0.262      0.339        695
  TIME                    1.000      0.060      0.113       1120
  EMAIL                   0.842      0.618      0.713       1116
  PHONE                   0.327      0.453      0.380       1059
  ADDRESS                 0.607      0.281      0.384       1632
  LOCATION                0.446      0.374      0.406       1178
  SSN                     0.564      0.318      0.407       1020
  UNIQUE_ID               0.000      0.000      0.000        723
  LICENSE_NUMBER          0.571      0.005      0.010        812
  IP_ADDRESS     

#### Finetuned 3k Results

In [47]:
ft_3k_tag_results, ft_3k_overall, ft_3k_sample_details, ft_3k_counts_dict = evaluate_token_level(ft_3k_val_results)

Token-level (per-tag) Performance
(each PII token counted individually)
Tag                   Precision     Recall         F1    Support
-----------------------------------------------------------------
  NAME                    0.940      0.964      0.952       1118
  TITLE                   0.862      0.968      0.912        437
  DOB                     0.964      0.977      0.970        959
  DATE                    0.925      0.934      0.929        695
  TIME                    0.919      0.963      0.941       1120
  EMAIL                   0.985      0.991      0.988       1116
  PHONE                   0.991      0.920      0.954       1059
  ADDRESS                 0.972      0.971      0.971       1632
  LOCATION                0.957      0.971      0.964       1178
  SSN                     0.892      0.880      0.886       1020
  UNIQUE_ID               0.749      0.975      0.847        723
  LICENSE_NUMBER          0.970      0.919      0.944        812
  IP_ADDRESS     

#### Finetuned 20k Results

In [48]:
ft_20k_tag_results, ft_20k_overall, ft_20k_sample_details, ft_20k_counts_dict = evaluate_token_level(ft_20k_val_results)

Token-level (per-tag) Performance
(each PII token counted individually)
Tag                   Precision     Recall         F1    Support
-----------------------------------------------------------------
  NAME                    0.982      0.977      0.979       1118
  TITLE                   0.973      0.982      0.977        437
  DOB                     0.979      0.964      0.971        959
  DATE                    0.927      0.945      0.936        695
  TIME                    0.977      0.964      0.971       1120
  EMAIL                   0.970      0.992      0.981       1116
  PHONE                   0.982      0.989      0.985       1059
  ADDRESS                 0.990      0.985      0.988       1632
  LOCATION                0.971      0.988      0.979       1178
  SSN                     0.985      0.987      0.986       1020
  UNIQUE_ID               0.941      0.974      0.957        723
  LICENSE_NUMBER          0.974      0.967      0.970        812
  IP_ADDRESS     

Once again, we see a significant improvement using the finetuned models rather that base Phi4. Specifically, recall is the best with the 20k finetuned. However, it is importanat to not that hallucinations are slightly higher for the 20k ft than 3k ft, perhaps due to the models trying too hard to find things to redact. Hallucinations were much higher for base Phi4.

### Evaluation Tier 2

Second, I ran the same evaluation scripts using the synthetic NP test transcripts. I expected these to perform worse since the were more dissimilar to the training data and also contained "fake" PII (details from a story recall about a Anna Thompson in South Boston).

In [57]:
warnings.filterwarnings('ignore')
base_np_silver_results = run_inference(np_silver_dataset, base_model, base_tokenizer, save_path="../data/base_np_silver.jsonl")
ft_3k_np_silver_results = run_inference(np_silver_dataset, ft_3k_model, ft_3k_tokenizer, save_path="../data/finetuned_3k_np_silver.jsonl")
ft_20k_np_silver_results = run_inference(np_silver_dataset, ft_20k_model, ft_20k_tokenizer, save_path="../data/finetuned_20k_np_silver.jsonl")

Resuming from record 100/100
Inference complete. Saved to ../data/base_np_silver.jsonl
Resuming from record 100/100
Inference complete. Saved to ../data/finetuned_3k_np_silver.jsonl
Resuming from record 100/100
Inference complete. Saved to ../data/finetuned_20k_np_silver.jsonl


### Calculate tag-level results

This is the same as what I did for the tier 1 dataset.

#### Base Phi4 Results

In [66]:
compute_tag_metrics(base_np_silver_results)

PER-TAG PERFORMANCE
Tag                   Precision     Recall         F1    Support
------------------------------------------------------------
  NAME                    1.000      1.000      1.000        100
  TITLE                   0.000      0.000      0.000          0
  DOB                     1.000      0.920      0.958        100
  DATE                    1.000      0.140      0.246        100
  TIME                    0.000      0.000      0.000          0
  EMAIL                   1.000      0.460      0.630        100
  PHONE                   1.000      0.630      0.773        100
  ADDRESS                 1.000      0.790      0.883        100
  LOCATION                1.000      0.890      0.942        100
  SSN                     0.000      0.000      0.000        100
  UNIQUE_ID               0.000      0.000      0.000          0
  LICENSE_NUMBER          0.000      0.000      0.000          0
  IP_ADDRESS              0.000      0.000      0.000          0
  USERNAM

defaultdict(<function __main__.compute_tag_metrics.<locals>.<lambda>()>,
            {'NAME': {'tp': 100, 'fp': 0, 'fn': 0},
             'DOB': {'tp': 92, 'fp': 0, 'fn': 8},
             'DATE': {'tp': 14, 'fp': 0, 'fn': 86},
             'EMAIL': {'tp': 46, 'fp': 0, 'fn': 54},
             'PHONE': {'tp': 63, 'fp': 0, 'fn': 37},
             'ADDRESS': {'tp': 79, 'fp': 0, 'fn': 21},
             'LOCATION': {'tp': 89, 'fp': 0, 'fn': 11},
             'SSN': {'tp': 0, 'fp': 0, 'fn': 100},
             'TITLE': {'tp': 0, 'fp': 0, 'fn': 0},
             'TIME': {'tp': 0, 'fp': 0, 'fn': 0},
             'UNIQUE_ID': {'tp': 0, 'fp': 0, 'fn': 0},
             'LICENSE_NUMBER': {'tp': 0, 'fp': 0, 'fn': 0},
             'IP_ADDRESS': {'tp': 0, 'fp': 0, 'fn': 0},
             'USERNAME': {'tp': 0, 'fp': 0, 'fn': 0},
             'PASS': {'tp': 0, 'fp': 0, 'fn': 0},
             'SEX': {'tp': 0, 'fp': 0, 'fn': 0},
             'CARDISSUER': {'tp': 0, 'fp': 0, 'fn': 0}})

#### Finetuned 3k Results

In [67]:
compute_tag_metrics(ft_3k_np_silver_results)

PER-TAG PERFORMANCE
Tag                   Precision     Recall         F1    Support
------------------------------------------------------------
  NAME                    1.000      0.460      0.630        100
  TITLE                   0.000      0.000      0.000          0
  DOB                     1.000      0.970      0.985        100
  DATE                    1.000      0.960      0.980        100
  TIME                    0.000      0.000      0.000          0
  EMAIL                   1.000      0.460      0.630        100
  PHONE                   1.000      0.490      0.658        100
  ADDRESS                 1.000      0.980      0.990        100
  LOCATION                1.000      0.750      0.857        100
  SSN                     0.000      0.000      0.000        100
  UNIQUE_ID               0.000      0.000      0.000          0
  LICENSE_NUMBER          0.000      0.000      0.000          0
  IP_ADDRESS              0.000      0.000      0.000          0
  USERNAM

defaultdict(<function __main__.compute_tag_metrics.<locals>.<lambda>()>,
            {'NAME': {'tp': 46, 'fp': 0, 'fn': 54},
             'DOB': {'tp': 97, 'fp': 0, 'fn': 3},
             'DATE': {'tp': 96, 'fp': 0, 'fn': 4},
             'EMAIL': {'tp': 46, 'fp': 0, 'fn': 54},
             'PHONE': {'tp': 49, 'fp': 0, 'fn': 51},
             'ADDRESS': {'tp': 98, 'fp': 0, 'fn': 2},
             'LOCATION': {'tp': 75, 'fp': 0, 'fn': 25},
             'SSN': {'tp': 0, 'fp': 0, 'fn': 100},
             'TIME': {'tp': 0, 'fp': 6, 'fn': 0},
             'UNIQUE_ID': {'tp': 0, 'fp': 26, 'fn': 0},
             'TITLE': {'tp': 0, 'fp': 3, 'fn': 0},
             'LICENSE_NUMBER': {'tp': 0, 'fp': 0, 'fn': 0},
             'IP_ADDRESS': {'tp': 0, 'fp': 0, 'fn': 0},
             'USERNAME': {'tp': 0, 'fp': 0, 'fn': 0},
             'PASS': {'tp': 0, 'fp': 0, 'fn': 0},
             'SEX': {'tp': 0, 'fp': 0, 'fn': 0},
             'CARDISSUER': {'tp': 0, 'fp': 0, 'fn': 0}})

#### Finetuned 20k Results

In [68]:
compute_tag_metrics(ft_20k_np_silver_results)

PER-TAG PERFORMANCE
Tag                   Precision     Recall         F1    Support
------------------------------------------------------------
  NAME                    1.000      0.110      0.198        100
  TITLE                   0.000      0.000      0.000          0
  DOB                     1.000      0.130      0.230        100
  DATE                    1.000      0.240      0.387        100
  TIME                    0.000      0.000      0.000          0
  EMAIL                   1.000      0.010      0.020        100
  PHONE                   1.000      0.130      0.230        100
  ADDRESS                 1.000      0.790      0.883        100
  LOCATION                1.000      0.290      0.450        100
  SSN                     0.000      0.000      0.000        100
  UNIQUE_ID               0.000      0.000      0.000          0
  LICENSE_NUMBER          0.000      0.000      0.000          0
  IP_ADDRESS              0.000      0.000      0.000          0
  USERNAM

defaultdict(<function __main__.compute_tag_metrics.<locals>.<lambda>()>,
            {'NAME': {'tp': 11, 'fp': 0, 'fn': 89},
             'DOB': {'tp': 13, 'fp': 0, 'fn': 87},
             'DATE': {'tp': 24, 'fp': 0, 'fn': 76},
             'EMAIL': {'tp': 1, 'fp': 0, 'fn': 99},
             'PHONE': {'tp': 13, 'fp': 0, 'fn': 87},
             'ADDRESS': {'tp': 79, 'fp': 0, 'fn': 21},
             'LOCATION': {'tp': 29, 'fp': 0, 'fn': 71},
             'SSN': {'tp': 0, 'fp': 0, 'fn': 100},
             'TITLE': {'tp': 0, 'fp': 0, 'fn': 0},
             'TIME': {'tp': 0, 'fp': 0, 'fn': 0},
             'UNIQUE_ID': {'tp': 0, 'fp': 0, 'fn': 0},
             'LICENSE_NUMBER': {'tp': 0, 'fp': 0, 'fn': 0},
             'IP_ADDRESS': {'tp': 0, 'fp': 0, 'fn': 0},
             'USERNAME': {'tp': 0, 'fp': 0, 'fn': 0},
             'PASS': {'tp': 0, 'fp': 0, 'fn': 0},
             'SEX': {'tp': 0, 'fp': 0, 'fn': 0},
             'CARDISSUER': {'tp': 0, 'fp': 0, 'fn': 0}})

Here, you can see that all three models performed horrifically, with extremely low recall in every category. Location and address interestingly suffered the least, but concerningly name, which is one of the most sensitive forms of PII, had an extrodinary low recall. This indicates that the fine-tuned model did not extrapolate well to the slightly different format of the NP transcripts.

### Token-level results

#### Base Phi4 Results

In [74]:
base_tag_results, base_overall, base_sample_details, base_counts_dict = evaluate_token_level(base_np_silver_results)

Token-level (per-tag) Performance
(each PII token counted individually)
Tag                   Precision     Recall         F1    Support
-----------------------------------------------------------------
  NAME                    0.527      0.337      0.411        522
  DOB                     0.356      0.453      0.399        340
  DATE                    0.062      0.002      0.004        447
  EMAIL                   0.918      0.225      0.361        400
  PHONE                   0.488      0.128      0.203        492
  ADDRESS                 0.716      0.256      0.377        939
  LOCATION                0.296      0.132      0.183        454
  SSN                     0.000      0.000      0.000        310
-----------------------------------------------------------------
Overall                   0.507      0.201      0.288       3904
Macro F1                  0.242

Hallucinated tokens (not in source): 34


#### Finetuned 3k Results

In [73]:
ft_3k_tag_results, ft_3k_overall, ft_3k_sample_details, ft_3k_counts_dict = evaluate_token_level(ft_3k_np_silver_results)

Token-level (per-tag) Performance
(each PII token counted individually)
Tag                   Precision     Recall         F1    Support
-----------------------------------------------------------------
  NAME                    0.698      0.169      0.272        522
  TITLE                   0.000      0.000      0.000          0
  DOB                     1.000      0.856      0.922        340
  DATE                    0.970      0.644      0.774        447
  TIME                    0.000      0.000      0.000          0
  EMAIL                   1.000      0.460      0.630        400
  PHONE                   1.000      0.287      0.445        492
  ADDRESS                 0.956      0.459      0.620        939
  LOCATION                0.463      0.251      0.326        454
  SSN                     0.000      0.000      0.000        310
  UNIQUE_ID               0.000      0.000      0.000          0
-----------------------------------------------------------------
Overall         

#### Finetuned 20k Results

In [72]:
ft_20k_tag_results, ft_20k_overall, ft_20k_sample_details, ft_20k_counts_dict = evaluate_token_level(ft_20k_np_silver_results)

Token-level (per-tag) Performance
(each PII token counted individually)
Tag                   Precision     Recall         F1    Support
-----------------------------------------------------------------
  NAME                    1.000      0.042      0.081        522
  DOB                     1.000      0.115      0.206        340
  DATE                    1.000      0.161      0.277        447
  EMAIL                   1.000      0.010      0.020        400
  PHONE                   1.000      0.065      0.122        492
  ADDRESS                 0.972      0.329      0.492        939
  LOCATION                0.691      0.084      0.149        454
  SSN                     0.000      0.000      0.000        310
-----------------------------------------------------------------
Overall                   0.952      0.132      0.232       3904
Macro F1                  0.168

Hallucinated tokens (not in source): 0


Here, we see that the precision was acceptable for the fintuned models where it significantly outperformed the base model. However, the recall is still incredibly low, which has negative implications for the usefulness of this model. Even if it is able to be precise, low recall means that a lot of tokens are being left caught in the text. One interesting finding is that again hallucination are much lower on the finetuned models, indicating that it is more faithful to the text being captured.

## Findings & Reflection

The fine-tuned Phi-4 model demonstrates strong generalization on held-out samples from the ai4privacy pii-masking-300k dataset, but there is a large performance gap when evaluated against the synthetic neuropsychological transcript set. All errors on the NP transcripts manifest as false negatives, meaning that the model misses PII entirely rather than hallucinating or misclassifying it.

#### Why the Model Performs Well on ai4privacy
The model was trained on ~3,000 examples drawn from pii-masking-300k, which consists of short, syntactically clean, declarative sentences with clearly delimited PII. The held-out eval slice comes from the same distribution, so strong performance here reflects the model learning the surface patterns of the training format rather than a deep understanding of PII in context. Sentences like "Contact John Smith at john@email.com" present PII in predictable, isolated positions that are relatively easy to detect.

#### Why the Model Struggles on NP Transcripts
The neuropsychological transcripts represent a different linguistic register and format that the model was not exposed to during training. I propose that three factors explain the majority of the performance gap:

1. Narrative embedding of PII (NAME, DOB/DATE)
In clinical transcripts, names and dates are embedded within long, complex sentences and clinical observations rather than presented as standalone facts. A sentence like "The patient, a 47-year-old male referred by Dr. Patel, presented on the fourteenth of March..." requires the model to resolve PII from discourse context. Short synthetic sentences have proven inadequate training data for developing this skill. High false negative rates on NAME and DOB/DATE suggest the model fails to recognize PII when it is not in a canonical position.
2. Clinical formatting of SSNs and identifiers
SSNs and patient identifiers in clinical documents appear in formats that differ from training examples. They are embedded in form-like structures, partially masked, or preceded by clinical shorthand (e.g. MRN:, DOB:, Pt. ID:).
3. Domain vocabulary and named entity ambiguity
Neuropsychological transcripts contain medication names, anatomical terms, and clinical scales (e.g. MMSE, WAIS) that share surface features with named entities. The model may suppress predictions in high-complexity clinical text due to uncertainty. This could mean that the omissions are caused by this uncertainty.

### Conclusion
The performance gap is likely due to the difference in data between a new domain. The model has learned the PII redaction task well within its training distribution. However, the 100-sample synthetic NP transcript set was constructed to surface the gap between the training data and its use in reality, and it succeeds in doing so. This eval set should be treated as the primary benchmark for future iterations, as it reflects the real deployment target.